In [1]:
import pandas as pd
import utils
from Bio import SeqIO
import pickle
import argparse
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from transformers import T5EncoderModel, T5Tokenizer
from config import config
from dataset import MyDataset
from torch import optim, nn
from torch.utils.data import TensorDataset
from torch.optim.lr_scheduler import ReduceLROnPlateau
from sklearn.ensemble import RandomForestClassifier
from model import Cnn, Cnn_muti
from torch.utils.data import DataLoader
from torch.optim import lr_scheduler
from utils import *
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, matthews_corrcoef
import numpy as np

In [2]:
utils.seed_everything(config.seed)

# Binary (PVP Indentification)

In [3]:
label_dict = {'non-PVP': 0, 'PVP': 1}
int2label = {0: 'non-PVP', 1: 'PVP'}

test_df = pd.read_csv(config.case_study_data)
test_proteins, test_ids = [], []
for index, row in test_df.iterrows():
    test_proteins.append(row['sequence'])
    test_ids.append(row['accession'])

test_labels = [max(label_dict.values()) + 1 for i in range(len(test_ids))]

y_test = np.array(test_labels)
test_data = MyDataset(test_ids, test_labels)
test_dataloader = DataLoader(test_data, shuffle=False, batch_size=config.batch_size)

In [4]:
# get embedding
embedding_df = pd.read_excel(config.embedding_file_case_study)
embedding_df.set_index("id", inplace=True)

In [5]:
model_2class = Cnn().to(config.device)
model_2class.load_state_dict(torch.load(config.binary_model))

/home/oyh/anaconda3/envs/pytorch/lib/python3.8/site-packages/torch/nn/modules/lazy.py:178: UserWarning: Lazy modules are a new feature under heavy development so changes to the API or functionality can happen at any moment.
  warnings.warn('Lazy modules are a new feature under heavy development '


<All keys matched successfully>

In [6]:
# test
_, test_epoch_preds = test(model_2class, test_dataloader, embedding_df)

# results
all_pred = [int2label[item] for item in test_epoch_preds]

new_df = pd.DataFrame({"accession":test_ids, "pred":all_pred})
new_df.to_csv("results/case_study/two_class_prediction.csv", index=False)

# Multi-class (PVP function annotation)

In [7]:
label_dict = {'minor capsid':0, 'tail fiber':1, 'major tail':2, 'portal':3, 'minor tail':4, 'baseplate':5, 'major capsid':6}
int2mutilabel = {0:'minor capsid', 1:'tail fiber', 2:'major tail', 3:'portal', 4:'minor tail', 5:'baseplate', 6:'major capsid'}

test_df = pd.read_csv(config.case_study_data)
test_proteins, test_ids, test_labels = [], [], []

for index, row in test_df.iterrows():
    test_proteins.append(row['sequence'])
    test_ids.append(row['accession'])

test_labels = [max(label_dict.values()) + 1 for i in range(len(test_ids))]
y_test = np.array(test_labels)
test_data = MyDataset(test_ids, test_labels)
test_dataloader = DataLoader(test_data, shuffle=False, batch_size=config.batch_size)

In [8]:
model_muti_class = Cnn_muti().to(config.device)
model_muti_class.load_state_dict(torch.load(config.muti_model))

/home/oyh/anaconda3/envs/pytorch/lib/python3.8/site-packages/torch/nn/modules/lazy.py:178: UserWarning: Lazy modules are a new feature under heavy development so changes to the API or functionality can happen at any moment.
  warnings.warn('Lazy modules are a new feature under heavy development '


<All keys matched successfully>

In [9]:
# test
_, test_epoch_preds, score = test_muti_score(model_muti_class, test_dataloader, embedding_df)

# results
all_pred = [int2mutilabel[item] for item in test_epoch_preds]

score_str = [str(s) for s in score]

new_df = pd.DataFrame({"accession":test_ids, "pred":all_pred, "score": score_str})
new_df.to_csv("results/case_study/muti_class_prediction.csv", index=False)